# ChurnSense-AI — 03 Feature Engineering

**Goal:** Transform cleaned customer data into **model-ready** numeric matrices for Phase 4 (training).

**Pipeline steps:**
1. Define features & target  
2. **Stratified train/test split** (before any fitting)  
3. **Encode categoricals** + **scale numerics** (fit on train only)  
4. **Handle class imbalance** with SMOTE (train only)  
5. Save artifacts for modeling  

---
Reusable code lives in `src/features.py` — the notebook explains **why** each step matters.

## 1. Setup

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT))

from src.features import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
    TEST_SIZE,
    apply_smote,
    load_clean_data,
    prepare_modeling_data,
    save_artifacts,
)

CLEAN_PATH = PROJECT_ROOT / "data" / "processed" / "telco_churn_clean.csv"
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

print(f"Project root: {PROJECT_ROOT}")

## 2. Load cleaned data

In [ ]:
df = load_clean_data(CLEAN_PATH, RAW_PATH)
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
df.head(2)

## 3. Define target and feature sets

| Role | Columns | Why |
|------|---------|-----|
| **Target** | `Churn` → 1 if Yes, 0 if No | Binary classification — predict leave vs stay |
| **Drop** | `customerID` | ID only — not predictive; would cause memorization |
| **Numeric** | tenure, MonthlyCharges, TotalCharges, SeniorCitizen | Continuous/count features → **scaled** |
| **Categorical** | Contract, PaymentMethod, services… | Text categories → **one-hot encoded** |

> **Paper alignment:** Chang et al. (2024) use a 75/25 stratified split and note ~2:1 non-churn:churn imbalance — we mirror the split ratio.

In [ ]:
y = (df[TARGET_COLUMN] == "Yes").astype(int)

print("Target distribution:")
print(y.value_counts().rename({0: "Stay (0)", 1: "Churn (1)"}))
print(f"\nChurn rate: {y.mean():.2%}")

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x=y.map({0: "Stay", 1: "Churn"}), palette=["#2ecc71", "#e74c3c"], ax=ax)
ax.set_title("Class imbalance — full dataset")
ax.set_xlabel("Label")
plt.tight_layout()
plt.show()

### Why imbalance matters

- With **~74% non-churn**, a naive model can hit high **accuracy** while missing most churners.
- Retention teams care about **recall** (catch people who will leave), not accuracy alone.
- Phase 4 will compare **baseline train** vs **SMOTE train** vs **`class_weight='balanced'`**.

## 4. Train / test split (stratified)

**Why split first?**  
If we encode or scale using the **full** dataset, information from the test set leaks into training. The model would look better than it truly is in production.

**Why stratify?**  
Keeps the same churn % in train and test — fair evaluation on both sets.

In [ ]:
bundle = prepare_modeling_data(df)

print(f"Train size: {len(bundle['y_train']):,} ({100 * (1 - TEST_SIZE):.0f}%)")
print(f"Test size:  {len(bundle['y_test']):,} ({100 * TEST_SIZE:.0f}%)")
print(f"Train churn rate: {bundle['y_train'].mean():.2%}")
print(f"Test churn rate:  {bundle['y_test'].mean():.2%}")
print(f"\nEncoded feature count: {bundle['X_train'].shape[1]}")

## 5. Categorical encoding (One-Hot)

**Why One-Hot Encoding (OHE)?**
- Converts categories like `Contract = Month-to-month` into 0/1 columns.
- **Logistic Regression** needs numeric inputs; OHE avoids false order (e.g. treating "Two year" as > "One year").
- **Tree models** (RF, XGBoost) also work well with OHE in a unified sklearn pipeline.

`handle_unknown='ignore'` → at scoring time, unseen categories get all-zero OHE columns (safe deployment).

In [ ]:
feature_names = bundle["feature_names"]
print(f"Total features after encoding: {len(feature_names)}")
print("\nSample feature names (first 15):")
for name in feature_names[:15]:
    print(f"  - {name}")

In [ ]:
# Show how Contract became multiple columns
contract_cols = [c for c in feature_names if "Contract" in c]
print("One-hot columns for Contract:")
print(contract_cols)

## 6. Feature scaling (StandardScaler)

**Why scale numeric features?**
- `TotalCharges` (thousands) vs `SeniorCitizen` (0/1) live on different scales.
- **Logistic Regression** uses gradient-based optimization — scaled features converge faster and weights are comparable.
- **Random Forest / XGBoost** are tree-based and mostly **scale-invariant**, but one shared pipeline keeps Phase 4 simple.

Formula: `z = (x - mean_train) / std_train` — means/stds computed **only on train**.

In [ ]:
X_train_df = pd.DataFrame(bundle["X_train"], columns=feature_names)

numeric_encoded = [c for c in feature_names if c.startswith("num__")]
print("Scaled numeric columns in matrix:")
print(numeric_encoded)

fig, ax = plt.subplots(figsize=(8, 4))
X_train_df[numeric_encoded].boxplot(ax=ax)
ax.set_title("Scaled numeric features (train set) — similar scales")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 7. Class imbalance — SMOTE on training data only

**Why SMOTE?**  
Synthetic Minority Over-sampling creates new **churn** examples between existing churners in feature space — the model sees more leave patterns.

**Why NOT on test data?**  
Test set must reflect **real-world** class balance so metrics are honest.

**Alternative (Phase 4):** `class_weight='balanced'` — no synthetic rows; penalizes missing churners during training.

In [ ]:
X_train_smote, y_train_smote = apply_smote(bundle["X_train"], bundle["y_train"])

bundle["X_train_smote"] = X_train_smote
bundle["y_train_smote"] = y_train_smote

print(f"Train rows before SMOTE: {len(bundle['y_train']):,}")
print(f"Train rows after SMOTE:  {len(y_train_smote):,}")
print(f"Churn rate before: {bundle['y_train'].mean():.2%}")
print(f"Churn rate after:  {y_train_smote.mean():.2%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, labels, title in zip(
    axes,
    [bundle["y_train"], y_train_smote],
    ["Train — original", "Train — after SMOTE"],
):
    counts = pd.Series(labels).value_counts().sort_index()
    sns.barplot(
        x=counts.index.map({0: "Stay", 1: "Churn"}),
        y=counts.values,
        palette=["#2ecc71", "#e74c3c"],
        ax=ax,
    )
    ax.set_title(title)
    ax.set_ylabel("Count")

plt.suptitle("Balancing strategy — training set only", y=1.02)
plt.tight_layout()
plt.show()

### Which training file to use in Phase 4?

| File | Use when |
|------|----------|
| `X_train.csv` + `y_train.csv` | Realistic imbalance; use with `class_weight='balanced'` |
| `X_train_smote.csv` + `y_train_smote.csv` | Compare SMOTE vs baseline (paper-style experiments) |
| `X_test.csv` + `y_test.csv` | **Always** use for evaluation — never SMOTE |

## 8. Quick sanity check — encoded matrix

In [ ]:
print("X_train shape:", bundle["X_train"].shape)
print("X_test shape:", bundle["X_test"].shape)
print("Any NaN in train?", np.isnan(bundle["X_train"]).any())
print("Any NaN in test?", np.isnan(bundle["X_test"]).any())

## 9. Save artifacts

In [ ]:
save_artifacts(bundle, OUTPUT_DIR, MODELS_DIR)

meta_path = OUTPUT_DIR / "feature_engineering_meta.json"
meta = json.loads(meta_path.read_text(encoding="utf-8"))

print("Saved files:")
for pattern in [
    "X_train.csv",
    "X_train_smote.csv",
    "X_test.csv",
    "y_train.csv",
    "y_train_smote.csv",
    "y_test.csv",
    "feature_engineering_meta.json",
]:
    path = OUTPUT_DIR / pattern
    print(f"  {'✓' if path.exists() else '✗'} {pattern}")

print(f"\nPreprocessor: {MODELS_DIR / 'preprocessor.joblib'}")
print(f"Features: {meta['n_features']}")

## 10. Phase 3 summary

| Step | Technique | Leakage-safe? |
|------|-----------|---------------|
| Split | 75/25 stratified | ✓ |
| Categorical | OneHotEncoder (fit on train) | ✓ |
| Numeric | StandardScaler (fit on train) | ✓ |
| Imbalance | SMOTE (train only) | ✓ |

**Outputs for modeling:**
- `data/processed/X_train.csv`, `y_train.csv` — baseline  
- `data/processed/X_train_smote.csv`, `y_train_smote.csv` — balanced train  
- `data/processed/X_test.csv`, `y_test.csv` — holdout evaluation  
- `models/preprocessor.joblib` — replay encoding for Streamlit / inference  
